# CTMAS — Federated Training on Colab (Free T4 GPU)

**Before running:** Runtime → Change runtime type → **T4 GPU**.

This notebook:
1. Mounts Google Drive (for `merged.csv` and saving the trained model)
2. Gets the code onto the Colab VM (GitHub clone OR zip upload)
3. Installs dependencies
4. Regenerates processed data from `merged.csv`
5. Runs 10 federated rounds on GPU
6. Saves `ctmas_model.pt` back to Drive

Expected wall-clock: ~15–30 min total on a free T4.

## 0. Sanity check — confirm GPU is attached

In [ ]:
import torch
print('CUDA available :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU            :', torch.cuda.get_device_name(0))
    print('VRAM (GB)      :', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
else:
    print('⚠️  No GPU. Runtime → Change runtime type → T4 GPU, then restart.')

## 1. Mount Google Drive

Upload `merged.csv` to your Drive first (anywhere — just remember the path).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Get the code onto the Colab VM

**Pick ONE of the two options below** (comment out the other).

### Option A — Clone from GitHub (recommended if you've pushed the repo)

In [ ]:
# Replace YOUR_USER/CTMAS with your actual GitHub path
# !git clone https://github.com/YOUR_USER/CTMAS.git /content/CTMAS
# %cd /content/CTMAS

### Option B — Upload a zip of the project folder

On your Mac: `cd ~/Developer && zip -r CTMAS.zip CTMAS -x 'CTMAS/Data/processed/*' -x 'CTMAS/Data/merged.csv' -x '*__pycache__*' -x '*.pt'`

Upload `CTMAS.zip` to Drive (smaller and faster than uploading the repo to the Colab file browser).

In [ ]:
# Adjust the zip path to where you put it in Drive
ZIP_PATH = '/content/drive/MyDrive/CTMAS.zip'

!rm -rf /content/CTMAS
!unzip -q "$ZIP_PATH" -d /content/
%cd /content/CTMAS
!ls

## 3. Copy `merged.csv` into the Data folder

Adjust the source path to wherever you put the CSV in Drive.

In [ ]:
CSV_IN_DRIVE = '/content/drive/MyDrive/merged.csv'   # ← edit if different

!mkdir -p Data
!cp "$CSV_IN_DRIVE" Data/merged.csv
!ls -lh Data/merged.csv

## 4. Install dependencies

Colab already ships `torch`, `pandas`, `scikit-learn`, `numpy`, `fastapi`. We only need the ML-specific extras.

In [ ]:
!pip install -q torch-geometric==2.6.1 flwr==1.15.0 opacus==1.5.2 shap uvicorn websockets

In [ ]:
# Quick import smoke test — fail fast if anything is wrong
import torch, torch_geometric, flwr, opacus
print('torch           :', torch.__version__)
print('torch_geometric :', torch_geometric.__version__)
print('flwr            :', flwr.__version__)
print('opacus          :', opacus.__version__)

## 5. Regenerate processed data

Takes ~2–3 min. Produces `X_train / X_val / X_test / y_test / metadata / scaler` in `Data/processed/`.

In [ ]:
%cd /content/CTMAS/Data
!python Data_Preprocessing.py
%cd /content/CTMAS

## 6. Train — 10 federated rounds on the T4

Tip: if you want a fast smoke test first, add `--subsample 0.1 --rounds 3`.

In [ ]:
!python main.py --rounds 10

## 7. Save the trained model back to Drive

So you don't lose it when the Colab runtime shuts down.

In [ ]:
import shutil, os, datetime
os.makedirs('/content/drive/MyDrive/CTMAS_artifacts', exist_ok=True)
stamp = datetime.datetime.now().strftime('%Y%m%d_%H%M')
dst = f'/content/drive/MyDrive/CTMAS_artifacts/ctmas_model_{stamp}.pt'
shutil.copy('/content/CTMAS/ctmas_model.pt', dst)
print('Saved to:', dst)

## 8. (Optional) Eval-only re-run with different detector settings

Useful if you want to tune thresholds later without retraining.

In [ ]:
!python main.py --eval-only

---

### Troubleshooting

- **`ModuleValidator` / `GradSampleModule` errors from Opacus** → torch/Opacus version mismatch. Try `!pip install -q opacus==1.5.1` or upgrade torch.
- **torch-geometric install fails** → run `!pip install torch-scatter torch-sparse -f https://data.pyg.org/whl/torch-$(python -c 'import torch; print(torch.__version__.split("+")[0])')+cu121.html` before installing torch-geometric.
- **No GPU assigned** → free-tier GPUs aren't guaranteed at peak hours. Try Runtime → Disconnect and delete runtime, then reconnect. Off-peak hours (late night UTC) are best.
- **Session disconnects mid-training** → save checkpoints more often, or run with `--rounds 5` twice and load the saved state in between.